In [3]:
import pandas as pd


feature_names = []
with open("./dataset_files/spambase.names", "r") as file:
    for line in file: 
        if line and not line.startswith("|") and ":" in line:
            feature_names.append(line.split(":")[0])

feature_names.append("is_spam")

df = pd.read_csv("./dataset_files/spambase.data", header=None, names=feature_names)

print("Shape:", df.shape)
df.head()

Shape: (4601, 58)


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [4]:
#Splitting features and labels
X = df.drop("is_spam", axis=1).values
y = df["is_spam"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
df["is_spam"].value_counts()


X shape: (4601, 57)
y shape: (4601,)


is_spam
0    2788
1    1813
Name: count, dtype: int64

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

#Logistical regressoion with scaling

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=2000)
)

#linear SVM
lsvm = make_pipeline(
    StandardScaler(),
    LinearSVC(random_state=2000)
)

#Random forrest
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=2000,
    n_jobs=1
)

models = {
    "Logistic Regression": lr, 
    "Linear SVM": lsvm, 
    "Random Forest": rf
}

models

{'Logistic Regression': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('logisticregression',
                  LogisticRegression(max_iter=1000, random_state=2000))]),
 'Linear SVM': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('linearsvc', LinearSVC(random_state=2000))]),
 'Random Forest': RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=2000)}

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import time

# Stratified 10-fold cross-validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=2000  # for reproducibility
)



In [7]:
def run_cv_evalutation(models, X,y,cv):

    results = {
        name: {"time": [], "acc": [], "f1": []}
        for name in models.keys()
    }

    for fold, (train_idx, test_idx) in enumerate(cv.split(X,y), start=1):
        print(f"Fold {fold} / {cv.n_splits}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for name, model in models.items():

            start = time.perf_counter()
            model.fit(X_train, y_train)
            train_time = time.perf_counter() - start
            y_pred = model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average="binary", pos_label=1)

            results[name]["time"].append(train_time)
            results[name]["acc"].append(acc)
            results[name]["f1"].append(f1)

    return results

In [8]:
cv_results = run_cv_evalutation(models, X, y, cv)

Fold 1 / 10
Fold 2 / 10
Fold 3 / 10
Fold 4 / 10
Fold 5 / 10
Fold 6 / 10
Fold 7 / 10
Fold 8 / 10
Fold 9 / 10
Fold 10 / 10


In [9]:
for name, res in cv_results.items():
    print(f"\nModel: {name}")
    print(f"  Mean train time: {np.mean(res['time']):.4f} s")
    print(f"  Mean accuracy:   {np.mean(res['acc']):.4f}")
    print(f"  Mean F1-score:   {np.mean(res['f1']):.4f}")



Model: Logistic Regression
  Mean train time: 0.0102 s
  Mean accuracy:   0.9257
  Mean F1-score:   0.9036

Model: Linear SVM
  Mean train time: 0.0462 s
  Mean accuracy:   0.9244
  Mean F1-score:   0.9018

Model: Random Forest
  Mean train time: 0.8463 s
  Mean accuracy:   0.9559
  Mean F1-score:   0.9434


In [10]:
def compute_ranks_for_measure(cv_results, measure, higher_is_better=True):
    model_names = list(cv_results.keys())
    n_models = len(model_names)
    n_folds = len(next(iter(cv_results.values()))[measure])

    ranks = {name: [] for name in model_names}

    for fold in range(n_folds):
        values = np.array([cv_results[name][measure][fold] for name in model_names])

        if higher_is_better: 
            order = np.argsort(-values)
        else:
            order = np.argsort(values)

        fold_ranks = np.empty(n_models)

        fold_ranks[order] = np.arange(1, n_models + 1)

        for i, name in enumerate(model_names):
            ranks[name].append(fold_ranks[i])

    return ranks

In [11]:
time_ranks = compute_ranks_for_measure(cv_results, "time", higher_is_better=False)
acc_ranks  = compute_ranks_for_measure(cv_results, "acc",  higher_is_better=True)
f1_ranks   = compute_ranks_for_measure(cv_results, "f1",   higher_is_better=True)


In [39]:
rows = []
model_names = list(cv_results.keys())
n_folds = len(next(iter(cv_results.values()))["acc"])

# Fold rows (values only)
for fold in range(n_folds):
    row = {"Fold": fold + 1}
    for name in model_names:
        row[name] = cv_results[name]["acc"][fold]
    rows.append(row)

# avg row
avg_row = {"Fold": "avg"}
for name in model_names:
    avg_row[name] = np.mean(cv_results[name]["acc"])
rows.append(avg_row)

# stdev row (sample stdev)
std_row = {"Fold": "stdev"}
for name in model_names:
    std_row[name] = np.std(cv_results[name]["acc"], ddof=1)
rows.append(std_row)

acc_table_124 = pd.DataFrame(rows)

# format to 4 decimals like the book
for name in model_names:
    acc_table_124[name] = acc_table_124[name].map(
        lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else x
    )

print("Table 2: Accuracy results (10-fold cross-validation)\n")
print(acc_table_124.to_markdown(index=False))


Table 2: Accuracy results (10-fold cross-validation)

| Fold   |   Logistic Regression |   Linear SVM |   Random Forest |
|:-------|----------------------:|-------------:|----------------:|
| 1      |                0.9414 |       0.9393 |          0.9761 |
| 2      |                0.9326 |       0.9261 |          0.9565 |
| 3      |                0.9239 |       0.9217 |          0.9674 |
| 4      |                0.9348 |       0.9391 |          0.9565 |
| 5      |                0.9261 |       0.9261 |          0.9478 |
| 6      |                0.9326 |       0.9283 |          0.9587 |
| 7      |                0.9348 |       0.9174 |          0.963  |
| 8      |                0.9174 |       0.9152 |          0.9435 |
| 9      |                0.9087 |       0.9174 |          0.9413 |
| 10     |                0.9043 |       0.913  |          0.9478 |
| avg    |                0.9257 |       0.9244 |          0.9559 |
| stdev  |                0.0121 |       0.0093 |          0.0

In [17]:
print("Table X: Accuracy results (10-fold cross-validation)")
print(acc_table.to_markdown(index=False))


Table X: Accuracy results (10-fold cross-validation)
| Fold    | Logistic Regression     | Linear SVM              | Random Forest           |
|:--------|:------------------------|:------------------------|:------------------------|
| 1       | 0.9414 (rank 2)         | 0.9393 (rank 3)         | 0.9761 (rank 1)         |
| 2       | 0.9326 (rank 2)         | 0.9261 (rank 3)         | 0.9565 (rank 1)         |
| 3       | 0.9239 (rank 2)         | 0.9217 (rank 3)         | 0.9674 (rank 1)         |
| 4       | 0.9348 (rank 3)         | 0.9391 (rank 2)         | 0.9565 (rank 1)         |
| 5       | 0.9261 (rank 2)         | 0.9261 (rank 3)         | 0.9478 (rank 1)         |
| 6       | 0.9326 (rank 2)         | 0.9283 (rank 3)         | 0.9587 (rank 1)         |
| 7       | 0.9348 (rank 2)         | 0.9174 (rank 3)         | 0.9630 (rank 1)         |
| 8       | 0.9174 (rank 2)         | 0.9152 (rank 3)         | 0.9435 (rank 1)         |
| 9       | 0.9087 (rank 3)         | 0.9174 (r

In [40]:
rows = []
model_names = list(cv_results.keys())
n_folds = len(next(iter(cv_results.values()))["time"])

# Fold rows
for fold in range(n_folds):
    row = {"Fold": fold + 1}
    for name in model_names:
        row[name] = cv_results[name]["time"][fold]
    rows.append(row)

# avg row
avg_row = {"Fold": "avg"}
for name in model_names:
    avg_row[name] = np.mean(cv_results[name]["time"])
rows.append(avg_row)

# stdev row (sample standard deviation, like the book)
std_row = {"Fold": "stdev"}
for name in model_names:
    std_row[name] = np.std(cv_results[name]["time"], ddof=1)
rows.append(std_row)

time_table_124 = pd.DataFrame(rows)

# format numbers to 4 decimals
for name in model_names:
    time_table_124[name] = time_table_124[name].map(
        lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else x
    )

print("Table 1: Training time results (10-fold cross-validation)\n")
print(time_table_124.to_markdown(index=False))


Table 1: Training time results (10-fold cross-validation)

| Fold   |   Logistic Regression |   Linear SVM |   Random Forest |
|:-------|----------------------:|-------------:|----------------:|
| 1      |                0.0146 |       0.0299 |          0.8532 |
| 2      |                0.0106 |       0.0865 |          0.8494 |
| 3      |                0.0093 |       0.0304 |          0.8327 |
| 4      |                0.0105 |       0.0429 |          0.8553 |
| 5      |                0.011  |       0.0401 |          0.8463 |
| 6      |                0.0096 |       0.0304 |          0.8476 |
| 7      |                0.0084 |       0.0434 |          0.8415 |
| 8      |                0.0085 |       0.0311 |          0.8482 |
| 9      |                0.0089 |       0.0335 |          0.8475 |
| 10     |                0.0102 |       0.0937 |          0.8414 |
| avg    |                0.0102 |       0.0462 |          0.8463 |
| stdev  |                0.0018 |       0.0238 |        

In [41]:
model_names = list(cv_results.keys())
n_folds = len(next(iter(cv_results.values()))["f1"])

rows = []
for fold in range(n_folds):
    row = {"Fold": fold + 1}
    for name in model_names:
        row[name] = cv_results[name]["f1"][fold]
    rows.append(row)

# avg row
avg_row = {"Fold": "avg"}
for name in model_names:
    avg_row[name] = np.mean(cv_results[name]["f1"])
rows.append(avg_row)

# stdev row (sample stdev like typical reporting: ddof=1)
std_row = {"Fold": "stdev"}
for name in model_names:
    std_row[name] = np.std(cv_results[name]["f1"], ddof=1)
rows.append(std_row)

f1_table_124 = pd.DataFrame(rows)

# format to 4 decimals like the book
for name in model_names:
    f1_table_124[name] = f1_table_124[name].map(lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else x)

print("Table 3: F-measure results (10-fold cross-validation)\n")
print(f1_table_124.to_markdown(index=False))
    

        

Table 3: F-measure results (10-fold cross-validation)

| Fold   |   Logistic Regression |   Linear SVM |   Random Forest |
|:-------|----------------------:|-------------:|----------------:|
| 1      |                0.9252 |       0.9227 |          0.97   |
| 2      |                0.9122 |       0.904  |          0.9448 |
| 3      |                0.9014 |       0.8977 |          0.9577 |
| 4      |                0.9176 |       0.9231 |          0.9448 |
| 5      |                0.9034 |       0.9034 |          0.9333 |
| 6      |                0.9136 |       0.9076 |          0.9474 |
| 7      |                0.9138 |       0.892  |          0.9524 |
| 8      |                0.8908 |       0.8883 |          0.9261 |
| 9      |                0.8814 |       0.8933 |          0.9256 |
| 10     |                0.8764 |       0.8857 |          0.9314 |
| avg    |                0.9036 |       0.9018 |          0.9434 |
| stdev  |                0.0161 |       0.0131 |          0.

### Friedman tests 
---

In [36]:


# Reset container
summary_rows = []

# Constants
N = 10
k = 3
df = k - 1
chi2_crit = 5.99

# Average ranks
avg_ranks_time = {name: np.mean(r) for name, r in time_ranks.items()}
avg_ranks_acc  = {name: np.mean(r) for name, r in acc_ranks.items()}
avg_ranks_f1   = {name: np.mean(r) for name, r in f1_ranks.items()}

# Friedman statistic
def friedman_chi2(avg_ranks_dict):
    R_bar = np.array(list(avg_ranks_dict.values()), dtype=float)
    return (12 * N / (k * (k + 1))) * (
        np.sum(R_bar**2) - (k * (k + 1)**2) / 4
    )

chi2_time = friedman_chi2(avg_ranks_time)
chi2_acc  = friedman_chi2(avg_ranks_acc)
chi2_f1   = friedman_chi2(avg_ranks_f1)

for measure_name, avg_ranks, chi2 in [
    ("Training time", avg_ranks_time, chi2_time),
    ("Accuracy", avg_ranks_acc, chi2_acc),
    ("F-measure", avg_ranks_f1, chi2_f1),
]:
    summary_rows.append({
        "Measure": measure_name,
        "avg rank (Logistic Regression)": avg_ranks["Logistic Regression"],
        "avg rank (Linear SVM)": avg_ranks["Linear SVM"],
        "avg rank (Random Forest)": avg_ranks["Random Forest"],
        "chi2_F": chi2,
        "df": df,
        "chi2_crit (0.05)": chi2_crit,
        "Reject H0?": "Yes" if chi2 > chi2_crit else "No"
    })

friedman_summary = pd.DataFrame(summary_rows)

friedman_summary



,Measure,avg rank (Logistic Regression),avg rank (Linear SVM),avg rank (Random Forest),chi2_F,df,chi2_crit (0.05),Reject H0?
0,Training time,1.0,2.0,3.0,20.0,2,5.99,Yes
1,Accuracy,2.3,2.7,1.0,15.8,2,5.99,Yes
2,F-measure,2.3,2.7,1.0,15.8,2,5.99,Yes


### Nemenyi Tests 
---

In [42]:
q_alpha = 2.343

CD = q_alpha * np.sqrt((k * (k+1)) / ( 6*N))
CD

np.float64(1.0478214542564015)

In [43]:
avg_ranks_acc = {
    "Random Forest": 1.0,
    "Logistical Regression": 2.3,
    "Linear SVM": 2.7
}

pairs = [
    ("Random Forest", "Logistical Regression"),
    ("Random Forest", "Linear SVM"),
    ("Logistical Regression", "Linear SVM")
]

for a, b in pairs:
    diff = abs(avg_ranks_acc[a] - avg_ranks_acc[b])
    print(f"{a} vs {b}: diff = {diff:.2f} -> {'Significant' if diff > CD else 'Not significant'}")

Random Forest vs Logistical Regression: diff = 1.30 -> Significant
Random Forest vs Linear SVM: diff = 1.70 -> Significant
Logistical Regression vs Linear SVM: diff = 0.40 -> Not significant


In [52]:
rows = []

for a, b in pairs:
    diff = abs(avg_ranks_acc[a] - avg_ranks_acc[b])
    rows.append({
        "Model A": a,
        "Model B": b,
        "Avg rank diff": diff,
        "Significant?": "Yes" if diff > CD else "No"
    })

pairwise_table = pd.DataFrame(rows)

latex = pairwise_table.to_latex(
    index=False,
    escape=False,
    float_format="%.2f",
    column_format="llcl"
)

print(latex)


\begin{tabular}{llcl}
\toprule
Model A & Model B & Avg rank diff & Significant? \\
\midrule
Random Forest & Logistical Regression & 1.30 & Yes \\
Random Forest & Linear SVM & 1.70 & Yes \\
Logistical Regression & Linear SVM & 0.40 & No \\
\bottomrule
\end{tabular}

